# Exercise: Logistic Regression

In [ ]:
import copy
from math import floor, ceil
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib as mpl
import matplotlib.pyplot as plt
from cycler import cycler
import seaborn as sns

# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

# Set default torch to torch.float64 to prevent NaN errors
torch.set_default_dtype(torch.float64)

In [ ]:
# Class that normalizes data to follow Normal(0, 1) distribution.
class normUnitvar:
    def __init__(self, fullDataset):
        self.normmean = fullDataset.mean(axis=0)
        self.normstd = fullDataset.std(axis=0)

    def normalize(self, data):
        return (data - self.normmean) / self.normstd

    def denormalize(self, data):
        return data * self.normstd + self.normmean

## Introduction
In this exercise, we will implement a Logistic Regression model, and work with a linear model, basis functions and finally a neural network. We will start with a 2-class problem ($\mathcal{C}_1$, $\mathcal{C}_2$). We scale the classes to be either 0 or 1, as we did in the lecture.

In [ ]:
# 1D target data [[x0, t0], [x1, t1], .. ]
data = torch.tensor(
    ([0.1, 1], [0.2, 1], [0.3, 1], [0.4, 1], [0.5, 0], [0.6, 0], [0.7, 0], [0.8, 0])
)
classes = 2
markers = ["o", "^", "s"]
nums = [1, 0]
label_names = ["$\mathcal{C}_1$", "$\mathcal{C}_2$", "$\mathcal{C}_3$"]

# Plot data
plt.figure(figsize=(6, 4))
for i in range(classes):
    plt.plot(
        data[data[:, 1] == nums[i]][:, 0],
        data[data[:, 1] == nums[i]][:, 1],
        markers[i],
        c=colors[i],
        fillstyle="none",
        label=label_names[i],
    )
    plt.legend()
plt.show()

## Linear Logistic Regression
Recall from the lecture `Logistic Regression` how we jump straight to inferring the posterior. 
For our 2-class problem this leads to

$$
p(\mathcal{C}_1 \vert\mathbf{x}) = \displaystyle\frac {1} {1+\exp(-a)} = \sigma(a) \quad \mathrm{with} \quad
a = \ln \frac{p(\boldsymbol{x} | \mathcal{C}_1) p(\mathcal{C}_1)}{p(\boldsymbol{x} | \mathcal{C}_2) p(\mathcal{C}_2)}
$$

which is known as the *logistic sigmoid function*.

Before using this in a model, let's implement this function and observe what it looks like.
Your task is to implement this sigmoid function function below.

In [ ]:
# Implement the sigmoid function
def sigmoid_func(a):
    """
    :return: the logistic sigmoid of a
    """

    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #

    return sigmoid


# Check if the sigmoid is implemented correctly
if torch.isclose(sigmoid_func(torch.tensor([0.5])), torch.tensor([0.62245933])):
    print("Sigmoid is implemented correctly")
else:
    print("Sigmoid is NOT implemented correctly")

# Plot the resulting figure
x = torch.arange(-10, 10, 0.1)
plt.figure(figsize=(4, 3))
plt.plot(x, sigmoid_func(x), label="Sigmoid", c=colors[0])
plt.legend()
plt.show()

Two key properties of the sigmoid function are its smoothness and its function values which are strictly between 0 and 1. 
The smoothness is useful for gradient descent, which we will use to find the parameters of our model.
Having values between 0 and 1 is a standard categorization for classification. 

Let's now use this logistic sigmoid function in a linear model:

$$
p(\mathcal{C}_1\vert\mathbf{x}) = \sigma\left(\mathbf{w}^T\mathbf{x} + w_0\right) \equiv y(\boldsymbol{x}).
$$

Classifying to $\mathcal{C}_1$ if $y(\mathbf{x})\geq 0.5$ and to $\mathcal{C}_2$ otherwise lets us derive at the cross-entropy error function:

$$
E(\mathbf{w}) = -\ln p(\mathcal{D}\vert\mathbf{w}) = \displaystyle -\sum_{n=1}^N\left[t_n\ln y_n + (1-t_n)\ln(1-y_n)\right].
$$

It is your task to implement this cross-entropy error function below.

In [ ]:
def cross_entropy(y, t):
    """
    :return: the cross-entropy (negative log likelihood)
    """

    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #

    return c_e


# Implementation check
if torch.isclose(
    cross_entropy(torch.tensor([0.2]), torch.tensor([1.0])),
    torch.tensor([1.60943]),
):
    print("Cross-entropy is implemented correctly")
else:
    print("Cross-entropy is NOT implemented correctly")

Use the code block below compare the cross-entropy for various predictions and targets. E.g., see what happens if you add another feature which is predicted exactly.

In [ ]:
# Example target
t_0 = torch.tensor([1.0])

# A prediction far from the target
y_0 = torch.tensor([0.2])
print(f"Cross-entropy for a bad prediction: {cross_entropy(y_0, t_0)}")

# A prediction closer to the target
y_1 = torch.tensor([0.95])
print(f"Cross-entropy for a good prediction: {cross_entropy(y_1, t_0)}")

# Your own tests:

With the cross-entropy in place, we are finally ready to create a linear model based on the logistic regression model and the sigmoid function.
This is implemented in the `linLogistic` class below. 
An additional `classify` function is added, which ensures that

$$
\text{if }y(\mathbf{x})\geq 0.5, \mathbf{x} \text{ belongs to } \mathcal{C}_1 \\
\text{if }y(\mathbf{x})<0.5, \mathbf{x} \text{ belongs to } \mathcal{C}_2
$$


In [ ]:
class linLogistic(nn.Module):
    def __init__(self, input_dim):
        super(linLogistic, self).__init__()

        torch.manual_seed(0)

        # Randomly initialize the parameters as "self.w"
        self.w = torch.rand(input_dim, requires_grad=True)

        return None

    def forward(self, x):
        # Compute the outputs based on x, the parameters and the sigmoid function
        outputs = sigmoid_func(torch.inner(self.w, x))

        return outputs.view(-1, 1)

    def classify(self, x):
        """
        Return the class label (0 or 1) for any input x, and the predicted output y.
        Note: make sure that the output y is detached (using '.detach()') from the computational graph, to prevent plotting issues.
        """

        y = self.forward(x).detach()
        y_class = torch.where(y > 0.5, 0.0, 1.0)

        return y_class, y

## Finding the model parameters
In the discriminant exercise, we could find the analytical least-squares solution.
By including the sigmoid in our model, it is no longer linear with respect to the parameters $\mathbf{w}$.
Therefore, we will instead use a gradient descent optimizer to find the parameters that minimize the cross-entropy.
You don't need to do anything in these two code blocks, they should be very familiar to you by now.

In [ ]:
# Find parameters using adam (a gradient descent based optimizer)

def optimParameters(model, train_loader, val_loader, lambda_val=0.01, n_epochs=2000):
    adam = torch.optim.Adam([model.w], lr=0.1)
    best_val_MSE = 1e10  # high enough to always be lowered in epoch 0

    for epoch in range(n_epochs):
        # Training
        for data in train_loader:  # loop over batches
            x, t = data
            y = model(x)

            # We add the L2 regularization:
            w = model.w
            L2_loss = lambda_val / 2 * torch.inner(w, w)
            loss = cross_entropy(y, t) / y.shape[0] + L2_loss

            adam.zero_grad()  # Reset gradients
            loss.backward()  # Backpropagation
            adam.step()  # Update parameters

        # Validation
        val_loss = 0
        for data in val_loader:
            with torch.no_grad():
                x, t = data
                y = model(x)
                val_loss += cross_entropy(y, t) / y.shape[0]

        # Is the current model better than the best model so far?
        if val_loss < best_val_MSE:
            best_model = copy.deepcopy(model)
            best_val_MSE = val_loss
            best_epoch = epoch

        # Has the best model not improved for 50 epochs?
        if epoch > best_epoch + 50:
            break

        if epoch % 200 == 0:
            print(f"Step: {epoch}, Current validation loss: {val_loss}")

    print(
        f"Final step: {epoch}, loss: {val_loss}, best model at epoch {best_epoch} with loss {best_val_MSE}"
    )
    return best_model

In [ ]:
# Split a dataset into a training and validation set. You can ignore this code block
def createDataLoaders(dataset, batch_size=8):
    # Split dataset into a training and validation set
    train_split = 0.8
    val_split = 0.2

    train_size = floor(train_split * len(dataset))
    val_size = ceil(val_split * len(dataset))

    # By using a generator, we ensure that the split is reproducible
    generator1 = torch.Generator().manual_seed(0)
    train_set, val_set = torch.utils.data.random_split(
        dataset, [train_size, val_size], generator=generator1
    )

    # Create dataloaders
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

We now use these functions to split our data, initialize our model and then find the parameters to fit the data. 

In [ ]:
# We add a bias term to out inputs as we are used to from regression
# Note that torch needs the output to be a 2D array as well
x = data[:, 0]
t = data[:, 1]
X = torch.concat((torch.ones((x.shape[0], 1)), x.view(-1, 1)), dim=1)
T = t.view(-1, 1)

# Create dataloaders
train_loader, val_loader = createDataLoaders(torch.utils.data.TensorDataset(X, T))

# Initialize the linear logistic regression model
LinLogReg = linLogistic(X.shape[1])

# Initialize a regularization value
lambda_val = 0.01

# Train the model
LinLogReg = optimParameters(LinLogReg, train_loader, val_loader, lambda_val)

## Plotting the result
We have created our 2-class logistic regression model and found the model parameters that minimize the validation loss for our simple example. 
We now can plot and analyze our model output.

In [ ]:
# Plotting data, ignore this code
def plotClassPreds(x_test, predictions, y, data, dx):
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))

    for i in range(x_test.shape[0] - 1):
        plt.fill_between(
            [x_test[i].item() - dx / 2, x_test[i + 1].item() - dx / 2],
            -1.1,
            1.1,
            color=colors[int(predictions[i].item())],
            alpha=0.2,
            edgecolor="none",
        )

    ax.plot(x_test, y, c="k", label="y(x)")

    for i in range(classes):
        ax.plot(
            data[data[:, 1] == nums[i]][:, 0],
            data[data[:, 1] == nums[i]][:, 1],
            markers[i],
            c=colors[i],
            fillstyle="none",
            label=label_names[i],
        )

    # Making the legend
    handles, labels = ax.get_legend_handles_labels()
    handles.extend(
        [
            mpl.patches.Patch(facecolor=colors[i], edgecolor="k", alpha=0.3)
            for i in range(classes)
        ]
    )
    labels.extend([f"Classified to {label_names[i]}" for i in range(classes)])
    ax.legend(handles=handles, labels=labels, fontsize=10, loc="center left")

    plt.xlim(torch.min(data[:, 0]) - 0.3, torch.max(data[:, 0]) + 0.3)
    plt.ylim(-0.1, 1.1)

    plt.xlabel("x")
    plt.ylabel("Class")
    plt.show()


# Create test data and plot
dx = 0.0025
x_test = torch.arange(torch.min(data[:, 0]) - 0.3, torch.max(data[:, 0]) + 0.3, dx)
X_test = torch.concat((torch.ones((x_test.shape[0], 1)), x_test.view(-1, 1)), dim=1)
predictions, y = LinLogReg.classify(X_test)

plotClassPreds(x_test, predictions, y, data, dx)

Notice that even though we added a non-linearity to our model by employing the sigmoid function, the decision boundary remains linear.

## Beyond linearly seperable data
In almost all practical cases, the data is not linearly seperable. 
We will now consider a slightly more complex problem which can no longer be linearly separated as shown below.

In [ ]:
# Creating data for a more complex case that can't be seperated linearly
def true_func(x):
    y_r = torch.sin(2.5 * torch.pi * x + 1) / 2 + 0.5
    y_c = [int(torch.floor(x) if x < 0.5 else torch.ceil(x)) for x in y_r]
    return torch.tensor(y_c)


x_complex = torch.linspace(0, 1, 21)
t_complex = true_func(x_complex)
complex_data = torch.stack((x_complex, t_complex), dim=1)

# Plot data
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
for i in range(classes):
    ax.plot(
        complex_data[complex_data[:, 1] == nums[i]][:, 0],
        complex_data[complex_data[:, 1] == nums[i]][:, 1],
        markers[i],
        c=colors[i],
        fillstyle="none",
        label=label_names[i],
    )
plt.show()

There are several ways to add non-linearity to our classification model. 
We will use two methods you're familiar with from the regression case here, namely basis functions and neural networks. 

## Basis-functions logistic regression
Instead of operating on our inputs $x$ directly, we use a set of basis functions $\bf{\phi}(\mathbf{x})$ in our model:

$$
y(\mathbf{x})=\sigma(\mathbf{w}^T\bf{\phi}(\mathbf{x})).
$$

It is your task to implement the basis function model.
We've initialized some of the code to work with radial basis functions, recall that it's function is

$$
\phi_j=\exp\left[ - \displaystyle\frac{(x-\mu_j)^2}{2\ell^2}\right]
$$

with $\mu_j$ the mean and $\ell$ the length scale of the basis function. 

In [ ]:
class basisLogistic(nn.Module):
    def __init__(self, domain, M_radial, l_radial):
        """
        :param domain: The boundaries (i.e. [0, 1]) of the means of the radial basis functions
        :param M_radial: The number of radial basis functions. These will be evenly spaced between the domain boundaries
        :param l_radial: The length scale of the radial basis functions, i.e. how wide they are
        """
        super(basisLogistic, self).__init__()

        self.domain = domain
        self.M_radial = M_radial
        self.l_radial = l_radial
        torch.manual_seed(0)

        # ---------------------- student exercise --------------------------------- #
        # YOUR CODE HERE
        # ---------------------- student exercise --------------------------------- #

    def forward(self, phi):
        # Compute the outputs of the model

        # ---------------------- student exercise --------------------------------- #
        # YOUR CODE HERE
        # ---------------------- student exercise --------------------------------- #

        return outputs.view(-1, 1)

    def basisFunc(self, x_input):
        # Compute Phi

        # ---------------------- student exercise --------------------------------- #
        # YOUR CODE HERE
        # ---------------------- student exercise --------------------------------- #

        return Phi

    def classify(self, x):
        """
        Return the class label (0 or 1) for any input x, and return the output of the forward function.
        Note: make sure that the output y is detached (using '.detach()') from the computational graph, to prevent plotting issues.
        """

        # ---------------------- student exercise --------------------------------- #
        # YOUR CODE HERE
        # ---------------------- student exercise --------------------------------- #

        return y_class, y

Now we can initialize our model and optimize its parameters, before plotting the result.

In [ ]:
BasisLogReg = basisLogistic([0, 1], 9, 0.1)

# Get the basis function matrix
Phi = BasisLogReg.basisFunc(x_complex)
T_complex = t_complex.view(-1, 1)

# Create the data loaders
train_loader, val_loader = createDataLoaders(
    torch.utils.data.TensorDataset(Phi, T_complex)
)

# Find the parameters
BasisLogReg = optimParameters(BasisLogReg, train_loader, val_loader, lambda_val)

In [ ]:
# Create test data
dx = 0.0025
x_test = torch.arange(torch.min(x_complex) - 0.3, torch.max(x_complex) + 0.3, dx)

# Initialize phi
Phi_test = BasisLogReg.basisFunc(x_test)

# classify test data
predictions, y = BasisLogReg.classify(Phi_test)

# plot
plotClassPreds(x_test, predictions, y, complex_data, dx)

In the first example, we observed the region outside of the data range to stay consistent with the closest data point. 
Here, we can see that the prediction in regions outside of the data range does not necessarily match the closest data point.
Why is this the case?
Is this always happen with linear basis function models?

## Logistic Regression with Neural Networks
So far, the result of logistic regression seems very similar to that of discriminant models, except that $y$ follows a more non-linear path and is limited to $[0,1]$.
We also no longer have an analytical solution.
Let's now extend our method to a case with 2D inputs and 3 classes that can't be perfectly separated, and use a neural network instead of the basis function model.
As usual with neural networks, we first have to standardize the input data.
To avoid cluttering the code we use the standardized dataset throughout the remainder of this notebook as if it was the original data.

In [ ]:
# Setup
classes = 3  # output classes
dimensions = 2  # input dimension
colors[2] = colors[3]

# IRIS DATASET
from sklearn import datasets

data = datasets.load_iris()
X = torch.tensor(data.data[:, :2])
t = torch.tensor(data.target)

# Create normalizers and normalize the data
x_normalizer = normUnitvar(X)
X_norm = x_normalizer.normalize(X)

# Plot data
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

for i in range(classes):
    ax.plot(
        X_norm[t == i][:, 0],
        X_norm[t == i][:, 1],
        markers[i],
        c=colors[i],
        fillstyle="none",
        label=label_names[i],
    )

ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
plt.legend()
plt.xlim(-3, 3)
plt.ylim(-3, 3)

# Reshape targets to be in form class1=[1,0,0], class3=[0, 0, 1] etc.
T_new = torch.zeros((t.shape[0], classes))
for i in range(t.shape[0]):
    T_new[i, int(t[i])] = 1

### Exercise: Implement the cross-entropy and softmax functions
The first thing to consider is that with three classes, having a single output is no longer sufficient.

Our cross-entropy error function defined before is no longer correct, so we need to implement a new function.
The cross-entropy for a multi-class problem is defined as:

$$
E(\mathbf{w}) = -\sum_{n=1}^N\sum_{k=1}^K t_{nk}\ln y_{nk}
$$

Furthermore, we will therefore use a softmax function to get a probability distribution over the classes.
Recall from the lecture that softmax is defined as:

$$
\mathrm{softmax}(\mathbf{a},a_k) = \frac{\exp(a_k)}{\sum_j\exp(a_j)}.
$$

Softmax converts an arbitraty vector into a vector of probabilities, where each element is in the range [0,1] and the sum of all elements is 1.
Implement the softmax function in the cell below.

In [ ]:
# Cross-entropy for >2 classes
def cross_entropy_multiclass(y, t):
    """
    :return: the cross-entropy (negative log likelihood)
    """
    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #
    return c_e


# Defining the softmax function
def softmax(values):
    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #

    return softmax


val1 = torch.tensor([0.1, 0.5, 0.9])
out1 = softmax(val1)
print(f"Input 1: {val1}\n Softmax 1: {out1}\n sum of softmax: {torch.sum(out1)}")
assert torch.isclose(
    out1[0], torch.tensor([0.211982])
), "Softmax is not correctly implemented"

val2 = torch.tensor([10, -5, 3, 4.1, 0])
out2 = softmax(val2)
print(f"\nInput 2: {val2}\n Softmax 2: {out2}\n sum of softmax: {torch.sum(out2)}")

Observe how the softmax scales the different original values, and how the sum of its entries is always 1.

### Exercise: Implement a neural network structure for classification

We now turn to neural networks to solve our 3-class classification problem. 
Implement a neural network structure in the code below.
Consider where the softmax function should be used, and how the discrete predicted class is obtained.

In [ ]:
# Implement a NN structure with the softmax LogisticRegression
# You are free to choose the number of hidden layers, nodes, and activation functions of the starting layers
class NNLogistic(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_nodes=10):
        super(NNLogistic, self).__init__()

        # ---------------------- student exercise --------------------------------- #
        # YOUR CODE HERE
        # ---------------------- student exercise --------------------------------- #

    def forward(self, x):
        """
        Pass the input through the network to get a probability distribution over the classes
        """

        # ---------------------- student exercise --------------------------------- #
        # YOUR CODE HERE
        # ---------------------- student exercise --------------------------------- #

        return outputs

    def classify(self, x):
        """
        Return:
            y_class: the class label (0, 1, or 2) for any input x
            y: the output of the forward function.
        Note: make sure that the output y is detached (using '.detach()') from the computational graph, to prevent plotting issues.
        """

        # ---------------------- student exercise --------------------------------- #
        # YOUR CODE HERE
        # ---------------------- student exercise --------------------------------- #

        return y_class, y

## Regularization
Before fitting our model to the data, we need to consider regularization.
The maximum likelihood approach we are using can exhibit severe overfitting, where large weights lead to a flexibel model with high variance that is likely to overfit. 
It is, therefore, good practice to regularize the model. 
One method of regularization is to punish the model for having large weights, such as by adding the L2 norm  of the weight vector  $L_2=\frac{\lambda}{2}\mathbf{w}^T\mathbf{w}$ to the training loss.
Up to this point in the notebook, we have carried out the L2-regularization with a fixed value $\lambda=0.01$.

Now, let's study its effect and find an optimal value.
We choose to redefine our training loop to incorporate this, but note that most of this training loop is the same as before.

In [ ]:
def optimParameters_L2(model, train_loader, val_loader, lambda_val=0.01, n_epochs=2000):
    adam = torch.optim.Adam(
        model.parameters(), lr=0.01
    )  # We do not add weight_decay to adam here
    best_val_MSE = 1e10

    for epoch in range(n_epochs):
        # Training
        for data in train_loader:
            x, t = data
            y = model(x)
            # We add the L2 loss:
            w = torch.nn.utils.parameters_to_vector(model.parameters())
            L2_loss = lambda_val / 2 * torch.inner(w, w)
            loss = cross_entropy_multiclass(y, t) / y.shape[0] + L2_loss

            adam.zero_grad()
            loss.backward()
            adam.step()

        # Validation
        val_loss = 0
        for data in val_loader:
            with torch.no_grad():
                x, t = data
                y = model(x)
                val_loss += cross_entropy_multiclass(y, t) / y.shape[0]

        if val_loss < best_val_MSE:
            best_model = copy.deepcopy(model)
            best_val_MSE = val_loss
            best_epoch = epoch

        if epoch > best_epoch + 50:
            break

        if epoch % 200 == 0:
            print(f"Step: {epoch}, Current validation loss: {val_loss}")

    print(
        f"Final step: {epoch}, loss: {val_loss}, best model at epoch {best_epoch} with loss {best_val_MSE}"
    )
    return best_model, best_val_MSE

### Hyperparameter tuning
The degree of regularization is controlled by the parameter $\lambda$. To find a suitable value for $\lambda$, we can train different models with various $\lambda$ values and compare their losses. The model with the lowest error is then used for predictions.

In [ ]:
lambda_values = [0.00, 0.005, 0.01, 0.02, 0.05, 0.1]

# We now set the weight_decay to 0
lambda_weight_decay = 0.0

train_loader, val_loader = createDataLoaders(
    torch.utils.data.TensorDataset(X_norm, T_new)
)
min_loss = 1e9
best_lambda = 0.00

for lambda_val in lambda_values:
    # Initialize model
    NNLogReg = NNLogistic(dimensions, classes)
    w = torch.nn.utils.parameters_to_vector(NNLogReg.parameters())

    # Train model
    NNLogReg, val_loss = optimParameters_L2(
        NNLogReg, train_loader, val_loader, lambda_val
    )

    print(f"model with lambda = {lambda_val} has loss {val_loss}\n")
    # Save best model
    if val_loss < min_loss:
        min_loss = val_loss
        best_lambda = lambda_val
        NNLogReg_best = copy.deepcopy(NNLogReg)
print(f"Best model for lambda={best_lambda} with loss {min_loss}.")

## Plotting the results

With the model selection complete we can now plot our model prediction over the domain.

In [ ]:
# Creating a grid of test points
a = torch.arange(-3, 3.04, 0.04)
grid_len = len(a)
grid_x, grid_y = torch.meshgrid(a, a, indexing="ij")

# Flatten arrays and concatenate as 2 columns
x_test = torch.concat((grid_x.reshape(-1, 1), grid_y.reshape(-1, 1)), dim=1)

outputs = torch.empty((x_test.shape[0]))

for i in range(x_test.shape[0]):
    outputs[i] = NNLogReg_best.classify(x_test[i])[0]

"""
Plotting
"""
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

levels = [0, 0.5, 1, 2]
confill = plt.contourf(
    grid_x.numpy(),
    grid_y.numpy(),
    outputs.reshape((grid_len, grid_len)),
    levels,
    colors=colors,
    alpha=0.3,
)
levels = [0.5, 1]
plt.contour(grid_x.numpy(), grid_y.numpy(), outputs.reshape((grid_len, grid_len)), levels, colors="k")

# Targets
for i in range(classes):
    ax.plot(
        X_norm[t == i][:, 0],
        X_norm[t == i][:, 1],
        markers[i],
        c=colors[i],
        fillstyle="none",
        label=label_names[i],
    )
plt.legend()
plt.grid(False)
plt.xlim(-3, 3)
plt.ylim(-3, 3)
plt.show()

For a well-trained model, most of the observations should be classified correctly. 
Still, the overlap between $\mathcal{C}_2$ & $\mathcal{C}_3$ makes it impossible to get all the points right.

## Review

In this notebook you've become familiar with the concepts of the sigmoid function, cross-entropy, and softmax. 
Each of these are fundamental in classification, and we've built different models based on them.
In addition to these concepts, it is also important to reflect on how setting up the labels differs from regression cases.